In [22]:
from ollama import Client

model='llama3.1:8b'
client = Client(host='http://localhost:11434')

In [23]:
# Test ollama client
response = client.chat(
    model=model,
    messages=[
        {
            'role': 'user',
            'content': 'Привет! Напиши одно короткое предложение.',
        },
    ],
)
print(response['message']['content'])

Привет! Летом у нас часто жарко.


In [24]:
import re
import json

EXPECTED_FIELDS = {
    "product_id",
    "description",
    "pros",
    "cons",
    "tags",
}

MIN_DESCRIPTION_LENGTH = 120 
MAX_DESCRIPTION_LENGTH = 700 


class ResponseValidationError(ValueError):
    """Ошибка парсинга или проверки ответа модели."""


def count_sentences(text):
    """Число предложений в тексте. Точка внутри числа концом предложения не считается."""
    return len([s for s in re.split(r"[.!?]+(?!\d)", text) if s.strip()]) 


def parse_and_validate(content: str, product_id_req: str) -> dict[str, str]:
    """Преобразовать JSON-ответ и проверить его контракт."""

    ret = {"product_id": product_id_req}

    try:
        data = json.loads(content)
    except json.JSONDecodeError as error:
        return ret | {'error': f"Ответ не является корректным JSON: {error.msg}"}

    if not isinstance(data, dict):
        return ret | {'error': "На верхнем уровне должен находиться JSON-объект"}

    received_fields = set(data)
    missing_fields = EXPECTED_FIELDS - received_fields
    unexpected_fields = received_fields - EXPECTED_FIELDS

    if missing_fields:
        return ret | {'error': "Отсутствуют обязательные поля: "
            f"{sorted(missing_fields)}"
        }

    if unexpected_fields:
        return ret | {'error': "Отсутствуют обязательные поля: "
            "Получены лишние поля: "
            f"{sorted(unexpected_fields)}"
        }

    product_id = data['product_id']
    if product_id != product_id_req:
        return ret | {'error': "Поле product_id не соответствует входным параметрам"}

    description = data["description"]

    if not isinstance(description, str):
        return ret | {'error': "Поле description должно быть строкой"}

    description = description.strip()

    if not description:
        return ret | {'error': "Поле description не должно быть пустым"}
      
    i = count_sentences(description)
    if i < 2 or i > 5:
        return ret | {'error': "Description must contain 2-5 sentences"}

    if len(description) < MIN_DESCRIPTION_LENGTH:
        return ret | {'error': "Отсутствуют обязательные поля: "
            "Поле description содержит больше "
            f"{MAX_DESCRIPTION_LENGTH} символов"
        }

    if len(description) > MAX_DESCRIPTION_LENGTH:
        return ret | {'error': "Поле description содержит меньше "
            f"{MAX_DESCRIPTION_LENGTH} символов"
        }

    pros = data['pros']
    if len(pros) < 2 or len(pros) > 5:
       return ret | {'error': "Поле pros must contain 2-5 values"}

    cons = data['cons']
    if len(cons) < 1 or len(cons) > 3:
       return ret | {'error': "Поле cons must contain 1-3 values"}
    
    tags = data['tags']
    if len(tags) < 3 or len(tags) > 8:
       return ret | {'error': "Поле tags must contain 3-8 values"}
    
    return ret

In [25]:
from typing import List
from typing_extensions import Annotated

from pydantic import (
    AfterValidator,
    BaseModel,
    ConfigDict,
    Field,
    field_validator,
)

class CardRequest(BaseModel):
    """Product Card"""

    model_config = ConfigDict(extra="forbid")

    product_id: str = Field(min_length=1, max_length=50)
    description: Annotated[str, Field(min_length=120, max_length=700), AfterValidator(count_sentences)]
    pros: Annotated[List[str], Field(min_length=2, max_length=5)]
    cons: Annotated[List[str], Field(min_length=1, max_length=3)]
    tags: Annotated[List[str], Field(min_length=3, max_length=8)]


    @field_validator("description")
    @classmethod
    def validate_description(cls, value: str) -> str:
      """Удалить внешние пробелы и проверить описание."""
      value = value.strip()

      if not value:
        raise ValueError(
            "Поле summary не должно быть пустым"
        )
      
      i = count_sentences(value)
      if i < 2 or i > 5:
        raise ValueError("Description must contain 2-5 sentences")
      
      return value


    @field_validator('pros')
    @classmethod
    def check_pros(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы pros не могут быть пустыми строками")
      return v


    @field_validator('cons')
    @classmethod
    def check_cons(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы cons не могут быть пустыми строками")
      return v


    @field_validator('tags')
    @classmethod
    def check_tags(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы tags не могут быть пустыми строками")
      return v

In [26]:
response_schema = CardRequest.model_json_schema()
response_schema

{'additionalProperties': False,
 'description': 'Product Card',
 'properties': {'product_id': {'maxLength': 50,
   'minLength': 1,
   'title': 'Product Id',
   'type': 'string'},
  'description': {'maxLength': 700,
   'minLength': 120,
   'title': 'Description',
   'type': 'string'},
  'pros': {'items': {'type': 'string'},
   'maxItems': 5,
   'minItems': 2,
   'title': 'Pros',
   'type': 'array'},
  'cons': {'items': {'type': 'string'},
   'maxItems': 3,
   'minItems': 1,
   'title': 'Cons',
   'type': 'array'},
  'tags': {'items': {'type': 'string'},
   'maxItems': 8,
   'minItems': 3,
   'title': 'Tags',
   'type': 'array'}},
 'required': ['product_id', 'description', 'pros', 'cons', 'tags'],
 'title': 'CardRequest',
 'type': 'object'}

In [46]:
def request_one(
    msg: list[dict], 
    params={"temperature": 0.3, "top_p": 0.9, "top_k": 50, "repetition_penalty": 1.1, "max_tokens": 350}
) -> tuple[str, int]:
  messages=[
    {"role": "system", "content": 
      """
        Generate a product card in Russian (utf-8) by example.
        Example:
        {
          "product_id": "P-1001",
          "description": "Компактные внутриканальные наушники с Bluetooth 5.3 и защитой IPX4. Держат заряд до 24 часов и надёжно сидят при активном движении. Подойдут для спорта и повседневного использования.",
          "pros": ["Чистый звук", "Удобная посадка", "Долгое время работы"],
          "cons": ["Быстро разряжаются на морозе"],
          "tags": ["наушники", "беспроводные", "спорт", "IPX4", "Bluetooth 5.3"]
        } 
        "description": 2-5 sentences,
        return only JSON
      """
    } 
  ] + msg
  # print('>>> messages', messages)

  response = client.chat(
      model=model,  
      messages=messages,
      options={
        'seed':               42,  
        'temperature':        params['temperature'],  
        'top_p':              params['top_p'],
        'max_tokens':         params['max_tokens'],   
        "top_k":              params['top_k'], 
        "repetition_penalty": params['repetition_penalty'],      
      },
      format=CardRequest.model_json_schema(),
  )

  content = response['message']['content']
  if not content:
    raise ResponseValidationError(
        "Empty response of model"
    )

  output_tokens = response.eval_count
  
  return content, output_tokens

In [45]:
# test request_one
item = {"product_id": "P-0026", "title": "Смартфон Grafo G9", "category": "Электроника / Смартфоны", "attributes": {"бренд": "Grafo", "экран": "6.7\"", "память": "256 ГБ", "оперативная память": "8 ГБ", "аккумулятор": "6000 мА·ч", "основная камера": "50 Мп"}, "reviews": ["Пользуюсь пару недель. Камера снимает достойно даже вечером", "Брал для себя. Звонки чистые, собеседника слышно хорошо", "В целом Быстро заряжается, за час почти полный", "Заказывал на подарок. Работает шустро, приложения не тормозят", "Брал для себя. Камера мылит фото при недостатке света", "Пользуюсь пару недель. К вечеру заряд просаживается быстрее ожидаемого", "Пользуюсь пару недель. Экран яркий, на солнце всё видно", "Пользуюсь пару недель. Экран яркий, на солнце всё видно", "Пользуюсь каждый день. Экран яркий, на солнце всё видно", "Быстро заряжается, за час почти полный", "В руке лежит удобно, не скользит", "Динамик тихий, в шумном месте плохо слышно", "Работает шустро, приложения не тормозят", "В руке лежит удобно, не скользит", "Коротко: Работает шустро, приложения не тормозят", "Купил на замену старому. Обычный телефон на каждый день, без сюрпризов", "Брал для себя. Батарея держит полтора дня без подзарядки", "В целом Работает шустро, приложения не тормозят"]}

row_dict = [{"role": "user", "content": f"""{item}"""}]

resp, tokens = request_one(row_dict)
print('>>> resp\n', resp, tokens)

>>> resp
 {
  "product_id": "P-0026",
  "description": "Смартфон Grafo G9 - это мощный и функциональный смартфон с большим экраном 6,7\" и мощным аккумулятором 6000 мА·ч. Он оснащен 50-мегапиксельной основной камерой и 8 ГБ оперативной памяти. Смартфон Grafo G9 идеально подходит для каждого дня, он быстро заряжается и имеет длинное время работы.",
  "pros": ["Быстро заряжается", "Экран яркий", "Удобно лежит в руке", "Работает шустро", "Приложения не тормозят"],
  "cons": ["Динамик тихий", "Батарея быстро разряжается на морозе"],
  "tags": ["Смартфон", "Графо", "Электроника", "Смартфоны", "Большой экран", "Питание"]
} 245


In [47]:
MAX_ATTEMPTS = 3 

def generate_one(
    msg: list[dict], 
    product_id: str,
    params={"temperature": 0.3, "top_p": 0.9, "top_k": 50, "repetition_penalty": 1.1, "max_tokens": 350},
) -> tuple[str, int]:
  messages_history = msg.copy()

  counter = 0
  errors = 0
  for attempt in range(1, MAX_ATTEMPTS + 1):
    counter += 1
    content, tokens = request_one(messages_history, params)

    error_dict = parse_and_validate(content, product_id)
    if not 'error' in error_dict:
      return content, tokens
    else:
      errors += 1
      print('>>> error: ', error_dict)
      print('>>> error content: ', content)

      if attempt == MAX_ATTEMPTS:
        return {"product_id": product_id, "error": "ResponseValidationError"}

      messages_history = messages_history + [
        {
          "role": "assistant",
          "content": content,
        },
        {
          "role": "user",
          "content": (
              "Предыдущий ответ не прошёл проверку. "
              f"Причина: {error_dict}. "
              "Исправь ответ и верни только JSON"
          ),
        }
      ]

In [48]:
# test generate_one
item = {
  "product_id": "P-0027", 
  "title": "Пылесос Klin Eco", 
  "category": "Бытовая техника - Пылесосы", "attributes": {"бренд": "Klin", "тип": "с контейнером", "мощность всасывания": "220 Вт", "объём контейнера": "2 л", "время работы": "60 мин", "вес": "1.5 кг"}, 
  "reviews": ["Что могу сказать: Щётка забивается волосами, чистить неудобно", 
                "Насадки удобные, под мебель достаёт легко", 
  ]
}
row_dict = [{"role": "user", "content": f"""{item}"""}]

resp, tokens = generate_one(row_dict, product_id=item['product_id'])
print('>>> resp\n', resp, tokens)

>>> resp
 {
  "product_id": "P-0027",
  "description": "Пылесос Klin Eco - компактный и мощный пылесос с контейнером объемом 2 л. Обладает мощностью всасывания 220 Вт и может работать до 60 минут. Весит всего 1,5 кг, что делает его идеальным выбором для бытового использования.",
  "pros": ["Удобные насадки", "Легко подходит под мебель"],
  "cons": ["Щетка забивается волосами", "Чистить неудобно"],
  "tags": ["пылесос", "клин", "бытовая техника", "с контейнером", "220 Вт"]
} 172


In [49]:
file_name = './dev.jsonl'
with open(file_name, 'r', encoding='utf-8') as file:
  items = file.read().splitlines()

In [50]:
import json
from datetime import datetime
from pathlib import Path

def wright_files(timestamp_str_dir, results, params, df):
  timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
  directory = Path(f"./outputs/{timestamp_str_dir}/{timestamp_str}")
  directory.mkdir(parents=True, exist_ok=True)

  with open(directory / "results.json", "w", encoding="utf-8") as file:
    json.dump(results, file, indent=4, ensure_ascii=False)

  with open(directory / "params.json", "w", encoding="utf-8") as file:
    json.dump(params, file, indent=4, ensure_ascii=False)

  totals_path = f"./outputs/{timestamp_str_dir}" / "totals.csv"
  df.to_csv(totals_path, index=False, encoding="utf-8")

In [63]:
import tiktoken

def token_count_tiktoken(text: str) -> int:
  # Llama 3 использует токенизатор, очень похожий на cl100k_base или o200k_base
  encoding = tiktoken.get_encoding("cl100k_base")
  return len(encoding.encode(text))

In [64]:
# test token_count_tiktoken
tokens = token_count_tiktoken('hello world!')
tokens


SSLError: HTTPSConnectionPool(host='openaipublic.blob.core.windows.net', port=443): Max retries exceeded with url: /encodings/cl100k_base.tiktoken (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))

In [65]:
import os
from dotenv import load_dotenv
from transformers import AutoTokenizer

load_dotenv(override=True)
api_key = os.getenv("HF_TOKEN")
if not api_key:
    raise RuntimeError(
        "Переменная окружения API_KEY не задана"
    )

def token_count_trans(text: str) -> int:
  tokenizer = AutoTokenizer.from_pretrained(
    "unsloth/llama-3.1-8b-Instruct", 
    token=api_key,
    force_download=True
  )
  return len(tokenizer.encode(text))

In [66]:
tokens = token_count_trans('hello world!')
tokens

OSError: Can't load tokenizer for 'unsloth/llama-3.1-8b-Instruct'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'unsloth/llama-3.1-8b-Instruct' is the correct path to a directory containing all relevant files for a TokenizersBackend tokenizer.

In [60]:
import shutil
import os

# Путь к локальному кэшу Hugging Face по умолчанию
cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
bad_model_dir = os.path.join(cache_dir, "models--unsloth--llama-3.1-8b-Instruct")

if os.path.exists(bad_model_dir):
    shutil.rmtree(bad_model_dir)
    print("Поврежденный кэш успешно удален!")
else:
    print("Папка кэша не найдена или уже удалена.")

Поврежденный кэш успешно удален!


In [62]:
tokens = token_count_trans('hello world!')
tokens

OSError: Can't load tokenizer for 'unsloth/llama-3.1-8b-Instruct'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'unsloth/llama-3.1-8b-Instruct' is the correct path to a directory containing all relevant files for a TokenizersBackend tokenizer.

In [52]:
import json

MAX_TOKENS = 1500

def item_normalize(item: str):
  reviews = item["reviews"]
  while token_count('[' + ", ".join(reviews) + ']') > MAX_TOKENS:
     reviews.pop() 

  item["reviews"] = reviews
  return item

In [19]:
item = {"product_id": "P-0026", "title": "Смартфон Grafo G9", "category": "Электроника / Смартфоны", "attributes": {"бренд": "Grafo", "экран": "6.7\"", "память": "256 ГБ", "оперативная память": "8 ГБ", "аккумулятор": "6000 мА·ч", "основная камера": "50 Мп"}, "reviews": ["Пользуюсь пару недель. Камера снимает достойно даже вечером", "Брал для себя. Звонки чистые, собеседника слышно хорошо", "В целом Быстро заряжается, за час почти полный", "Заказывал на подарок. Работает шустро, приложения не тормозят", "Брал для себя. Камера мылит фото при недостатке света", "Пользуюсь пару недель. К вечеру заряд просаживается быстрее ожидаемого", "Пользуюсь пару недель. Экран яркий, на солнце всё видно", "Пользуюсь пару недель. Экран яркий, на солнце всё видно", "Пользуюсь каждый день. Экран яркий, на солнце всё видно", "Быстро заряжается, за час почти полный", "В руке лежит удобно, не скользит", "Динамик тихий, в шумном месте плохо слышно", "Работает шустро, приложения не тормозят", "В руке лежит удобно, не скользит", "Коротко: Работает шустро, приложения не тормозят", "Купил на замену старому. Обычный телефон на каждый день, без сюрпризов", "Брал для себя. Батарея держит полтора дня без подзарядки", "В целом Работает шустро, приложения не тормозят"]}

item_norm = item_normalize(item)
print('>>> item_norm', item_norm)

SSLError: HTTPSConnectionPool(host='openaipublic.blob.core.windows.net', port=443): Max retries exceeded with url: /encodings/cl100k_base.tiktoken (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))

In [14]:
import pandas as pd
import time
import json

items_slice = items[:1]

df = pd.DataFrame(columns=["num", "temperature", "top_p", "top_k", "repetition_penalty", "max_tokens", 'error', 'descr_len', 'tokens', 'time_per_item'])
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

counter = 0

# for temperature in [0.0, 0.3, 0.7]:
#   for top_p in [0.9, 1.0]:
#     for top_k in [20, 50, None]:
#       for repetition_penalty in [1.0, 1.1, 1.2]:
#         for max_tokens in [250, 350, 500]:
for temperature in [0.3]:
  for top_p in [0.9]:
    for top_k in [50]:
      for repetition_penalty in [1.1]:
        for max_tokens in [350]:
          params = {
            "temperature": temperature, 
            "top_p": top_p, 
            "top_k": top_k, 
            "repetition_penalty": repetition_penalty, 
            "max_tokens": max_tokens,
          }
          for item in items_slice:
            err = 0
            counter += 1

            item_norm = item_normalize(item)
            print('>>> item_norm', item_norm)

            row_dict = {"role": "user", "content": item_norm}
            # print('>>> item', i)

            item_dict = json.loads(item)

            started = time.perf_counter()
            results = generate_one([row_dict], product_id=item_dict['product_id'], params=params)
            generation_time = time.perf_counter() - started

            if 'error' in results:
              err = 1
              descr_len = 0
              tokens = 0
            else:
              results_dict = json.loads(results)
              descr_len = len(results_dict['description'])
              tokens = token_count(results)
              
            new_row = {'num': counter} | params | {'error': err, 'descr_len': descr_len, 'tokens': tokens, 'time_per_item': round(generation_time, 2)}
            print(new_row)

            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
            # wright_files(timestamp_str, results, new_row, df)

ConnectionError: HTTPSConnectionPool(host='openaipublic.blob.core.windows.net', port=443): Max retries exceeded with url: /encodings/cl100k_base.tiktoken (Caused by NameResolutionError("HTTPSConnection(host='openaipublic.blob.core.windows.net', port=443): Failed to resolve 'openaipublic.blob.core.windows.net' ([Errno 11002] getaddrinfo failed)"))

In [34]:
df_sorted = df.sort_values(by='time_per_item')
df_sorted

,temperature,top_p,top_k,repetition_penalty,max_tokens,error,descr_len,tokens,time_per_item
320,0.0,0.9,50,1.2,350,0,265,229,0.03
327,0.0,0.9,50,1.2,350,0,144,185,0.03
324,0.0,0.9,50,1.2,350,0,205,239,0.03
347,0.0,0.9,50,1.2,500,0,144,185,0.03
316,0.0,0.9,50,1.2,250,0,163,213,0.03
...,...,...,...,...,...,...,...,...,...
9,0.0,0.9,20,1.0,250,0,237,224,4.04
7,0.0,0.9,20,1.0,250,0,144,185,4.26
1,0.0,0.9,20,1.0,250,0,257,237,6.26
5,0.0,0.9,20,1.0,250,0,153,264,6.31


In [35]:
totals_path = Path(f"./outputs/totals_final_{timestamp_str}.csv")
df_sorted.to_csv(totals_path, index=False, encoding="utf-8")